# Project: Competitive Intelligence Monitor
**Brief from Paras:**
Runs weekly. Reads competitor list from Notion, scans web per competitor,
summarises into Notion, posts highlights to Slack.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class CompetitiveIntelState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    competitors: list[str]
    findings: dict[str, str]
    notion_page_id: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# list_reader - Notion competitor list reader
list_reader_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
list_reader_agent = create_react_agent(llm, list_reader_tools, prompt='''Read the 'Competitors' Notion DB. Return list of competitor names.''')
def list_reader_node(state):
    result = list_reader_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='list_reader')]}


# scanner - Web scanner
scanner_tools = toolset.get_tools(actions=[Action.TAVILY_TAVILY_SEARCH, Action.TAVILY_TAVILY_EXTRACT])
scanner_agent = create_react_agent(llm, scanner_tools, prompt='''For one competitor, find launches, pricing changes, hiring, funding from past 7 days.''')
def scanner_node(state):
    result = scanner_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='scanner')]}


# writer - Notion summary writer
writer_tools = toolset.get_tools(actions=[Action.NOTION_CREATE_NOTION_PAGE, Action.NOTION_APPEND_TEXT_BLOCKS])
writer_agent = create_react_agent(llm, writer_tools, prompt='''Write findings into a new Notion page in 'Weekly CI' DB.''')
def writer_node(state):
    result = writer_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='writer')]}


# broadcaster - Slack broadcaster
broadcaster_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
broadcaster_agent = create_react_agent(llm, broadcaster_tools, prompt='''Post top 5 highlights to #competitive-intel.''')
def broadcaster_node(state):
    result = broadcaster_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='broadcaster')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if not state.get('competitors'): return {'next_worker': 'list_reader'}
    pending = [c for c in state['competitors'] if c not in state['findings']]
    if pending: return {'next_worker': 'scanner'}
    if state.get('notion_page_id') is None: return {'next_worker': 'writer'}
    return {'next_worker': 'broadcaster' if 'highlights posted' not in (state['messages'][-1].content.lower() if state['messages'] else '') else 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'writer', 'scanner', 'list_reader', 'broadcaster'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("list_reader", list_reader_node)
g.add_node("scanner", scanner_node)
g.add_node("writer", writer_node)
g.add_node("broadcaster", broadcaster_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "list_reader": "list_reader",
    "scanner": "scanner",
    "writer": "writer",
    "broadcaster": "broadcaster",
    "__end__": END,
})
g.add_edge("list_reader", "supervisor")
g.add_edge("scanner", "supervisor")
g.add_edge("writer", "supervisor")
g.add_edge("broadcaster", "supervisor")

conn = sqlite3.connect("04_competitive_intel.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '04_competitive_intel-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'messages': [HumanMessage(content='Run weekly competitive intelligence scan')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
